# M2-B2 — Audit éthique Athéna RH — phase SYNC binôme

> **Mission** : audit éthique complet du dataset Adult Income enrichi de
> commentaires manager. Datasheet duo signée. La phase d'anonymisation
> personnelle se fera en async (notebook séparé).

Binôme : `<prénom1>` + `<prénom2>` — Date : `<date>`

**Conventions** :
- `random_state=42` partout
- Pas de `print` (utiliser `display()` ou laisser la cellule retourner)
- `pathlib.Path` sur les chemins

## 0. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
DATA_DIR = Path("../data")
FULL_PATH = DATA_DIR / "adult_income_with_comments.csv"
SAMPLE_PATH = DATA_DIR / "audit_sample.csv"

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv(FULL_PATH)
print(f"Shape : {df.shape}")
df.head(3)

Shape : (32561, 16)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,manager_comments
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,RAS pour Alexandre Traore cette année. Manager...
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,Entretien annuel de Yves Traore : bon élément....
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,Alerte comportementale signalée par Gabriel de...


## 1. Audit qualité express (~15 min)

Le dataset Adult est plus propre que German Credit (peu de manquants).
Survol express ici — l'audit éthique est le cœur de M2-B2.

In [3]:
# Audit qualité express (manquants, doublons, cohérence cible)
display(df.info())

missing = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda x: (x["missing_count"] / len(df)).round(4))
    .sort_values("missing_count", ascending=False)
 )

display(missing.head(10))

duplicate_rows = df.duplicated().sum()
quality_kpis = pd.DataFrame(
    {
        "kpi": ["n_rows", "n_cols", "duplicate_rows", "target_unique_values"],
        "value": [len(df), df.shape[1], int(duplicate_rows), df["income"].nunique()],
    }
)
display(quality_kpis)

target_dist = (
    df["income"]
    .value_counts(normalize=True)
    .rename("share")
    .to_frame()
    .assign(share=lambda x: (100 * x["share"]).round(2))
 )
display(target_dist)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   age               32561 non-null  int64 
 1   workclass         30725 non-null  object
 2   fnlwgt            32561 non-null  int64 
 3   education         32561 non-null  object
 4   education_num     32561 non-null  int64 
 5   marital_status    32561 non-null  object
 6   occupation        30718 non-null  object
 7   relationship      32561 non-null  object
 8   race              32561 non-null  object
 9   sex               32561 non-null  object
 10  capital_gain      32561 non-null  int64 
 11  capital_loss      32561 non-null  int64 
 12  hours_per_week    32561 non-null  int64 
 13  native_country    31978 non-null  object
 14  income            32561 non-null  object
 15  manager_comments  32561 non-null  object
dtypes: int64(6), object(10)
memory usage: 4.0+ MB


None

,missing_count,missing_rate
occupation,1843,0.0566
workclass,1836,0.0564
native_country,583,0.0179
age,0,0.0000
fnlwgt,0,0.0000
education,0,0.0000
education_num,0,0.0000
marital_status,0,0.0000
relationship,0,0.0000
race,0,0.0000


,kpi,value
0,n_rows,32561
1,n_cols,16
2,duplicate_rows,0
3,target_unique_values,2


,share
income,
<=50K,75.92
>50K,24.08


### Synthèse qualité express

- Taille du dataset : **32 561 lignes** et **16 colonnes**.
- Doublons exacts : **0**.
- Valeurs manquantes concentrées sur 3 colonnes :
  - `occupation` : 1 843 (~5,66 %)
  - `workclass` : 1 836 (~5,64 %)
  - `native_country` : 583 (~1,79 %)
- Cible `income` binaire et déséquilibrée : **75,92 % `<=50K`** vs **24,08 % `>50K`**.

Conclusion : le dataset est globalement exploitable pour l'audit éthique, avec un point d'attention sur les NA de variables socio-professionnelles.

## 2. Audit éthique complet (~1 h)

Calcul du **disparate impact** sur 3 variables sensibles + 1 intersection.

In [ ]:
def disparate_impact(df: pd.DataFrame, sensible: str, target: str = "income",
                     positive: str = ">50K") -> tuple[float, pd.Series]:
    """Calcule le DI = SR_min / SR_max + retourne la série des SR par groupe."""
    sr = df.groupby(sensible)[target].apply(lambda x: (x == positive).mean())
    di = sr.min() / sr.max()
    return di, sr

In [ ]:
# TODO — DI sur sex
# di_sex, sr_sex = disparate_impact(df, 'sex')

In [ ]:
# TODO — DI sur race

In [ ]:
# TODO — DI sur native_country (agrégé en USA / non-USA pour simplifier)
# df['native_us'] = (df['native_country'] == 'United-States').map({True: 'USA', False: 'non-USA'})

### Intersectionnalité — DI sur le croisement sex × race

In [ ]:
# TODO — créer une colonne sex_race et calculer le DI
# df['sex_race'] = df['sex'].str.cat(df['race'], sep='_')

## 3. Visualisations (≥ 5)

Au moins 5 visualisations : distribution cible + DI par variable sensible +
crosstab intersection.

In [ ]:
# TODO — 5 visualisations claires

## 4. Verdict éthique (paragraphe markdown)

> Court paragraphe : « Quels biais structurels avons-nous détectés ? Quel
> est le DI le plus problématique ? »

À recopier dans `../datasheet.md` (section *Composition* / *Risques*).

## 5. Aperçu de la colonne `manager_comments` (pour préparer l'async)

In [ ]:
# Quelques exemples pour identifier les types de PII
for i in range(5):
    print(f'--- Exemple {i+1} ---')
    print(df['manager_comments'].iloc[i])
    print()

## 6. Datasheet binôme (à compléter dans `../datasheet.md`)

Reprendre la structure Gebru — 7 sections, 2 pages max. Signée duo
(« Auteurs : <prénom1>, <prénom2> » en haut).